# 26 · Apprentissage par renforcement — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch26_rl/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch26_rl/03_notebook.ipynb`, créée par `python tools/start_chapter.py 26`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 26.Q1 à 26.Q12, 26.R1 à 26.R3, 26.1 à 26.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | 26.13 à 26.18 | la boucle et Gymnasium, les retours, Flippers et le L-learning (*prochaine session de génération*) | | | |
| B | 26.19 à 26.21 | le cœur de ta librairie `rl.py` : politiques, mises à jour, boucle d'entraînement (*prochaine session de génération*) | | | |
| C | 26.22 à 26.27 | le Q-learning en action : FrozenLake, courbes, un 🐛, Flippers bruité, itération sur les valeurs, hyperparamètres (*prochaine session de génération*) | | | |
| D | 26.28 à 26.30 | SARSA contre le Q-learning, au bord de la falaise et sur Flippers (*prochaine session de génération*) | | | |
| E | 26.31 à 26.34 | le rejeu d'expérience, la conception des récompenses, un résultat honnête (*prochaine session de génération*) | | | |
| F | 26.35 à 26.38 | le morpion, CartPole, l'auto-apprentissage et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="26")  # your own library

## Objectifs et rappel express

- Manipuler la boucle agent-environnement avec Gymnasium, et calculer des retours actualisés.
- Écrire dans `mylearn.rl` un agent tabulaire complet (politiques, mises à jour du Q-learning et de SARSA, boucle d'entraînement, itération sur les valeurs, mémoire de rejeu) et le comparer sur Flippers, FrozenLake, CliffWalking, le morpion et CartPole.
- Rapporter un résultat de RL sur plusieurs graines.

**Rappel express.** $G_t = r_t + \gamma\,G_{t+1}$. Q-learning : $Q(s, a) \leftarrow Q(s, a) + \alpha\,[r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$, la cible vaut $r$ si $s'$ est terminal. SARSA : $Q(s', a')$ de l'action réellement choisie au lieu du max. ε-greedy : l'action gloutonne avec la probabilité $1 - \varepsilon + \varepsilon/|\mathcal{A}|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 26.1 à 26.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`), dans l'ordre demandé quand l'énoncé en impose un. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 26.7 et 26.8, l'estimation 🧮 26.9, la réflexion (🗣️ 26.10, ⚖️ 26.11, 📄 26.12) et l'entretien se corrigent avec `05_solutions.md`.

**26.Q1 — Apprendre avec des récompenses : ni supervisé, ni non supervisé**

In [ ]:
# 26.Q1: the values you found on paper
answer_26_Q1a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q1b = ...  # b) the letters, e.g. "AF"
answer_26_Q1c = ...  # c) True or False
answer_26_Q1d = ...  # d) True or False
answer_26_Q1e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q1a, answer_26_Q1b, answer_26_Q1c, answer_26_Q1d, answer_26_Q1e]):
    wb.check(f"26.Q1{letter}", answer)

**26.Q2 — Les quatre informations données à l'agent du morpion**

In [ ]:
# 26.Q2: the values you found on paper
answer_26_Q2a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q2b = ...  # b) the letters, e.g. "AF"
answer_26_Q2c = ...  # c) True or False
answer_26_Q2d = ...  # d) the letter of your choice, e.g. "F"
answer_26_Q2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_26_Q2a, answer_26_Q2b, answer_26_Q2c, answer_26_Q2d, answer_26_Q2e]):
    wb.check(f"26.Q2{letter}", answer)

**26.Q3 — État, actions, politique et informations privées**

In [ ]:
# 26.Q3: the values you found on paper
answer_26_Q3a = ...  # a) the letters, e.g. "AF"
answer_26_Q3b = ...  # b) the letter of your choice, e.g. "F"
answer_26_Q3c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q3d = ...  # d) True or False
answer_26_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_26_Q3a, answer_26_Q3b, answer_26_Q3c, answer_26_Q3d, answer_26_Q3e]):
    wb.check(f"26.Q3{letter}", answer)

**26.Q4 — Réponse de l'environnement, épisode et agents libres ou déclenchés**

In [ ]:
# 26.Q4: the values you found on paper
answer_26_Q4a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q4b = ...  # b) the letters, e.g. "AF"
answer_26_Q4c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q4d = ...  # d) True or False
answer_26_Q4e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q4a, answer_26_Q4b, answer_26_Q4c, answer_26_Q4d, answer_26_Q4e]):
    wb.check(f"26.Q4{letter}", answer)

**26.Q5 — MDP, observabilité, attribution du crédit, exploration ou exploitation**

In [ ]:
# 26.Q5: the values you found on paper
answer_26_Q5a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q5b = ...  # b) the letter of your choice, e.g. "F"
answer_26_Q5c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q5d = ...  # d) True or False
answer_26_Q5e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q5a, answer_26_Q5b, answer_26_Q5c, answer_26_Q5d, answer_26_Q5e]):
    wb.check(f"26.Q5{letter}", answer)

**26.Q6 — Retour total, retour actualisé et rôle de γ**

In [ ]:
# 26.Q6: the values you found on paper
answer_26_Q6a = ...  # a) a whole number
answer_26_Q6b = ...  # b) a number, 2 decimals
answer_26_Q6c = ...  # c) True or False
answer_26_Q6d = ...  # d) the letter of your choice, e.g. "F"
answer_26_Q6e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_26_Q6a, answer_26_Q6b, answer_26_Q6c, answer_26_Q6d, answer_26_Q6e]):
    wb.check(f"26.Q6{letter}", answer)

**26.Q7 — Flippers et L-learning : pourquoi « lousy » ?**

In [ ]:
# 26.Q7: the values you found on paper
answer_26_Q7a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q7b = ...  # b) a whole number
answer_26_Q7c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q7d = ...  # d) True or False
answer_26_Q7e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q7a, answer_26_Q7b, answer_26_Q7c, answer_26_Q7d, answer_26_Q7e]):
    wb.check(f"26.Q7{letter}", answer)

**26.Q8 — La mise à jour du Q-learning : α, γ et la valeur de l'état suivant**

In [ ]:
# 26.Q8: the values you found on paper
answer_26_Q8a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q8b = ...  # b) the letter of your choice, e.g. "F"
answer_26_Q8c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q8d = ...  # d) True or False
answer_26_Q8e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_26_Q8a, answer_26_Q8b, answer_26_Q8c, answer_26_Q8d, answer_26_Q8e]):
    wb.check(f"26.Q8{letter}", answer)

**26.Q9 — ε-greedy ou softmax : vrai ou faux**

In [ ]:
# 26.Q9: the values you found on paper
answer_26_Q9a = ...  # a) True or False
answer_26_Q9b = ...  # b) a number, 2 decimals
answer_26_Q9c = ...  # c) True or False
answer_26_Q9d = ...  # d) True or False
answer_26_Q9e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q9a, answer_26_Q9b, answer_26_Q9c, answer_26_Q9d, answer_26_Q9e]):
    wb.check(f"26.Q9{letter}", answer)

**26.Q10 — Table initialisée à zéro : comment la valeur remonte et converge**

In [ ]:
# 26.Q10: the values you found on paper
answer_26_Q10a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q10b = ...  # b) the letter of your choice, e.g. "F"
answer_26_Q10c = ...  # c) True or False
answer_26_Q10d = ...  # d) the letter of your choice, e.g. "F"
answer_26_Q10e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q10a, answer_26_Q10b, answer_26_Q10c, answer_26_Q10d, answer_26_Q10e]):
    wb.check(f"26.Q10{letter}", answer)

**26.Q11 — SARSA contre Q-learning : l'action suivante réellement jouée**

In [ ]:
# 26.Q11: the values you found on paper
answer_26_Q11a = ...  # a) the letter of your choice, e.g. "F"
answer_26_Q11b = ...  # b) the letter of your choice, e.g. "F"
answer_26_Q11c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q11d = ...  # d) True or False
answer_26_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_26_Q11a, answer_26_Q11b, answer_26_Q11c, answer_26_Q11d, answer_26_Q11e]):
    wb.check(f"26.Q11{letter}", answer)

**26.Q12 — Tables géantes, rejeu d'expérience, AlphaGo Zero et lancer de rayons**

In [ ]:
# 26.Q12: the values you found on paper
answer_26_Q12a = ...  # a) a whole number
answer_26_Q12b = ...  # b) the letter of your choice, e.g. "F"
answer_26_Q12c = ...  # c) the letter of your choice, e.g. "F"
answer_26_Q12d = ...  # d) True or False
answer_26_Q12e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_Q12a, answer_26_Q12b, answer_26_Q12c, answer_26_Q12d, answer_26_Q12e]):
    wb.check(f"26.Q12{letter}", answer)

**26.R1 — Un réseau comme fonction qui remplace une table (le décodeur d'un VAE)**

In [ ]:
# 26.R1: the values you found on paper
answer_26_R1a = ...  # a) the letter of your choice, e.g. "F"
answer_26_R1b = ...  # b) True or False
answer_26_R1c = ...  # c) a whole number

for letter, answer in zip("abc", [answer_26_R1a, answer_26_R1b, answer_26_R1c]):
    wb.check(f"26.R1{letter}", answer)

**26.R2 — Que sauvegarder pour reprendre un entraînement ?**

In [ ]:
# 26.R2: the values you found on paper
answer_26_R2a = ...  # a) the letters, e.g. "AF"
answer_26_R2b = ...  # b) the letter of your choice, e.g. "F"
answer_26_R2c = ...  # c) True or False

for letter, answer in zip("abc", [answer_26_R2a, answer_26_R2b, answer_26_R2c]):
    wb.check(f"26.R2{letter}", answer)

**26.R3 — Bandit à bras multiples, ε-greedy et UCB**

In [ ]:
# 26.R3: the values you found on paper
answer_26_R3a = ...  # a) the letter of your choice, e.g. "F"
answer_26_R3b = ...  # b) a number, 2 decimals
answer_26_R3c = ...  # c) the letter of your choice, e.g. "F"

for letter, answer in zip("abc", [answer_26_R3a, answer_26_R3b, answer_26_R3c]):
    wb.check(f"26.R3{letter}", answer)

**Ex 26.1 — Retours total et actualisé d'une partie de six coups**

In [ ]:
# Ex 26.1: the values you found on paper
answer_26_1a = ...  # a) a whole number
answer_26_1b = ...  # b) a whole number
answer_26_1c = ...  # c) a number, 3 decimals
answer_26_1d = ...  # d) a number, 3 decimals
answer_26_1e = ...  # e) a number, 2 decimals
answer_26_1f = ...  # f) a whole number
answer_26_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_26_1a, answer_26_1b, answer_26_1c, answer_26_1d, answer_26_1e, answer_26_1f, answer_26_1g]):
    wb.check(f"26.1{letter}", answer)

**Ex 26.2 — Taille des tables : de Flippers au Go**

In [ ]:
# Ex 26.2: the values you found on paper
answer_26_2a = ...  # a) a whole number
answer_26_2b = ...  # b) a whole number
answer_26_2c = ...  # c) a whole number
answer_26_2d = ...  # d) a whole number
answer_26_2e = ...  # e) a whole number
answer_26_2f = ...  # f) a whole number
answer_26_2g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_26_2a, answer_26_2b, answer_26_2c, answer_26_2d, answer_26_2e, answer_26_2f, answer_26_2g]):
    wb.check(f"26.2{letter}", answer)

**Ex 26.3 — Une mise à jour de Q-learning à la main**

In [ ]:
# Ex 26.3: the values you found on paper
answer_26_3a = ...  # a) a number, 2 decimals
answer_26_3b = ...  # b) a number, 2 decimals
answer_26_3c = ...  # c) a number, 2 decimals
answer_26_3d = ...  # d) a number, 2 decimals
answer_26_3e = ...  # e) a number, 2 decimals
answer_26_3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_26_3a, answer_26_3b, answer_26_3c, answer_26_3d, answer_26_3e, answer_26_3f]):
    wb.check(f"26.3{letter}", answer)

**Ex 26.4 — La même transition vue par SARSA**

In [ ]:
# Ex 26.4: the values you found on paper
answer_26_4a = ...  # a) a number, 2 decimals
answer_26_4b = ...  # b) a number, 3 decimals
answer_26_4c = ...  # c) a number, 3 decimals
answer_26_4d = ...  # d) True or False
answer_26_4e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_26_4a, answer_26_4b, answer_26_4c, answer_26_4d, answer_26_4e]):
    wb.check(f"26.4{letter}", answer)

**Ex 26.5 — Probabilités d'action : ε-greedy (standard et version du livre) et softmax**

In [ ]:
# Ex 26.5: the values you found on paper
answer_26_5a = ...  # a) a number, 2 decimals
answer_26_5b = ...  # b) a number, 2 decimals
answer_26_5c = ...  # c) a number, 2 decimals
answer_26_5d = ...  # d) a number, 3 decimals
answer_26_5e = ...  # e) a number, 3 decimals
answer_26_5f = ...  # f) a number, 3 decimals
answer_26_5g = ...  # g) True or False
answer_26_5h = ...  # h) a number, 3 decimals

for letter, answer in zip("abcdefgh", [answer_26_5a, answer_26_5b, answer_26_5c, answer_26_5d, answer_26_5e, answer_26_5f, answer_26_5g, answer_26_5h]):
    wb.check(f"26.5{letter}", answer)

**Ex 26.6 — Regarder la valeur remonter une chaîne d'états**

In [ ]:
# Ex 26.6: the values you found on paper
answer_26_6a = ...  # a) a whole number
answer_26_6b = ...  # b) a number, 2 decimals
answer_26_6c = ...  # c) a number, 2 decimals
answer_26_6d = ...  # d) a whole number
answer_26_6e = ...  # e) a number, 3 decimals
answer_26_6f = ...  # f) a number, 3 decimals
answer_26_6g = ...  # g) a number, 2 decimals
answer_26_6h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_26_6a, answer_26_6b, answer_26_6c, answer_26_6d, answer_26_6e, answer_26_6f, answer_26_6g, answer_26_6h]):
    wb.check(f"26.6{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire et appliquer à la main les mises à jour du Q-learning et de SARSA, état terminal compris ?
2. Sais-tu expliquer ce que règlent $\alpha$, $\gamma$ et $\varepsilon$, et ce qui se passe quand on les choisit mal ?
3. Sais-tu dire pourquoi une table ne passe pas à l'échelle, et ce que DQN y ajoute ?

**Pour aller plus loin** : les chapitres 2, 3 et 6 de Sutton et Barto (2018), cités dans la fiche. La suite : le ch. 27, les GAN.

*Les exercices de code 26.13 à 26.38 (parties A à F : la boucle et Flippers, ta librairie `rl.py`, le Q-learning, SARSA, le rejeu et les récompenses, le morpion, CartPole et l'auto-apprentissage) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 26` pour obtenir le notebook complet.*